# VRTPP-PR: Paper Model Implementation

This notebook is a close implementation of the model from:

> **"Optimal Routing and Trajectory Planning for Asteroid Mining with Partial In-Situ Resource Utilization"**  
> Euihyeon Choi and Koki Ho, Georgia Institute of Technology, AIAA SciTech 2026

The formulation follows the paper's MINLP/MILP/NLP structure. Two implementation guards are documented in the code: ending-base indices are kept disjoint from starting-base indices, and arcs between duplicate nodes of the same physical body are filtered out because they create artificial zero-physics hops.

## Algorithm (Sec. IV)
Iterative MILP-NLP decomposition until delta-v matrix converges (Eq. 47, epsilon_c = 10^-3):
1. **Init** (Sec. IV.A): trust-region NLP from T_d=0 using Hohmann half-period and full-period transfer-time seeds per body pair -> initial mhat_ij matrix
2. **MILP** (Sec. IV.B.1, Eqs. 37-42): fix mhat_ij, solve for optimal visiting sequence x, refueling r, cargo y
3. **NLP** (Sec. IV.B.2, Eqs. 43-45): fix sequence, minimise delta-v per arc via trust-region (T_j^a = T_i^{d*} + T_{ij}^{t*}); first-seen asteroid-to-asteroid arcs also try a few short local wait seeds to preserve the paper-route basin
4. Update mhat_ij = exp(-delta-v* / (g0 I_sp)), go to step 2

## Key equations implemented
| Component | Paper equations |
|---|---|
| Index sets | Eqs. 1-9 |
| MINLP objective | Eq. 10 |
| Network constraints | Eqs. 11-14 |
| Mass flow (MILP) | Eqs. 38-42 |
| Cumulative mining | Eqs. 20-22 |
| Propellant bounds | Eqs. 23-26 |
| y linearisation | Eqs. 33-34 |
| NLP (Lambert) | Eqs. 43-45 |
| Convergence criterion | Eq. 47 |
| Earth delta-v correction | Eq. 48 |

## Case Study (Sec. V.A, Tables 2-3)
- **Refueling asteroids:** Ryugu, Bennu (n_r = 2, n_rv = 3)
- **Mining asteroids:** 2001 SG10, 1989 ML, 1996 FG3, 2001 CC21, 1943 Anteros
- **Spacecraft:** m_dry = 300 kg, m_max = 20,000 kg, I_sp = 457 s, lambda = 5e-5 kg^-1
- **Expected result (Table 4):** 1 spacecraft, Earth -> FG3 -> Bennu -> Earth, ~10 iterations


## 1. Imports

In [ ]:
import numpy as np
from scipy.optimize import minimize, Bounds
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import time
import warnings
warnings.filterwarnings('ignore')

# Optimization
import gurobipy as gp
from gurobipy import GRB

## 2. Orbital Mechanics

The `OrbitalBody` class represents a celestial body with classical orbital elements and provides methods to compute position and velocity at any time using Kepler's equation.

The `LambertSolver` class solves Lambert's problem using universal variables with Stumpff functions, which is used to determine transfer trajectories between two positions in a given time of flight.

In [ ]:
class OrbitalBody:
    """Celestial body with orbital elements."""

    def __init__(self, name: str, a: float, e: float, i: float,
                 Omega: float, omega: float, M0: float, epoch: float = 0.0):
        """
        Parameters:
        -----------
        name : str - Body name
        a : float - Semi-major axis [AU]
        e : float - Eccentricity
        i : float - Inclination [degrees]
        Omega : float - RAAN [degrees]
        omega : float - Argument of periapsis [degrees]
        M0 : float - Mean anomaly at epoch [degrees]
        epoch : float - Reference epoch [TU]
        """
        self.name = name
        self.a = a
        self.e = e
        self.i = np.deg2rad(i)
        self.Omega = np.deg2rad(Omega)
        self.omega = np.deg2rad(omega)
        self.M0 = np.deg2rad(M0)
        self.epoch = epoch

    def position_at_time(self, t: float, mu: float = 1.0) -> np.ndarray:
        """Calculate position vector at time t using Kepler's equation."""
        n = np.sqrt(mu / self.a**3)
        M = self.M0 + n * (t - self.epoch)
        E = self._solve_kepler(M, self.e)
        nu = 2 * np.arctan2(np.sqrt(1 + self.e) * np.sin(E / 2),
                            np.sqrt(1 - self.e) * np.cos(E / 2))
        r_mag = self.a * (1 - self.e * np.cos(E))
        x_orb = r_mag * np.cos(nu)
        y_orb = r_mag * np.sin(nu)
        R = self._rotation_matrix()
        r_orb = np.array([x_orb, y_orb, 0])
        r = R @ r_orb
        return r

    def velocity_at_time(self, t: float, mu: float = 1.0) -> np.ndarray:
        """Calculate velocity vector at time t."""
        n = np.sqrt(mu / self.a**3)
        M = self.M0 + n * (t - self.epoch)
        E = self._solve_kepler(M, self.e)
        nu = 2 * np.arctan2(np.sqrt(1 + self.e) * np.sin(E / 2),
                            np.sqrt(1 - self.e) * np.cos(E / 2))
        h = np.sqrt(mu * self.a * (1 - self.e**2))
        vx_orb = -(mu / h) * np.sin(nu)
        vy_orb = (mu / h) * (self.e + np.cos(nu))
        R = self._rotation_matrix()
        v_orb = np.array([vx_orb, vy_orb, 0])
        v = R @ v_orb
        return v

    def _solve_kepler(self, M: float, e: float, tol: float = 1e-10) -> float:
        """Solve Kepler's equation using Newton-Raphson."""
        E = M if e < 0.8 else np.pi
        for _ in range(50):
            f = E - e * np.sin(E) - M
            f_prime = 1 - e * np.cos(E)
            E_new = E - f / f_prime
            if abs(E_new - E) < tol:
                return E_new
            E = E_new
        return E

    def _rotation_matrix(self) -> np.ndarray:
        """Compute rotation matrix from orbital plane to heliocentric frame."""
        c_O, s_O = np.cos(self.Omega), np.sin(self.Omega)
        c_i, s_i = np.cos(self.i), np.sin(self.i)
        c_w, s_w = np.cos(self.omega), np.sin(self.omega)
        R = np.array([
            [c_O * c_w - s_O * c_i * s_w, -c_O * s_w - s_O * c_i * c_w, s_O * s_i],
            [s_O * c_w + c_O * c_i * s_w, -s_O * s_w + c_O * c_i * c_w, -c_O * s_i],
            [s_i * s_w, s_i * c_w, c_i]
        ])
        return R

In [ ]:
class LambertSolver:
    """Robust Lambert solver using universal variables with Stumpff functions."""

    def __init__(self, mu: float = 1.0):
        self.mu = mu

    def solve(self, r1_vec: np.ndarray, r2_vec: np.ndarray, tof: float,
              prograde: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        """Solve Lambert's problem."""
        r1 = np.linalg.norm(r1_vec)
        r2 = np.linalg.norm(r2_vec)

        cos_dnu = np.dot(r1_vec, r2_vec) / (r1 * r2)
        cos_dnu = np.clip(cos_dnu, -1.0, 1.0)

        cross = np.cross(r1_vec, r2_vec)
        if prograde:
            if cross[2] >= 0:
                dnu = np.arccos(cos_dnu)
            else:
                dnu = 2 * np.pi - np.arccos(cos_dnu)
        else:
            if cross[2] < 0:
                dnu = np.arccos(cos_dnu)
            else:
                dnu = 2 * np.pi - np.arccos(cos_dnu)

        A = np.sin(dnu) * np.sqrt(r1 * r2 / (1 - cos_dnu))

        if abs(A) < 1e-14:
            raise ValueError("Degenerate Lambert problem")

        # Stumpff functions
        def C2(psi):
            if psi > 1e-6:
                return (1 - np.cos(np.sqrt(psi))) / psi
            elif psi < -1e-6:
                return (np.cosh(np.sqrt(-psi)) - 1) / (-psi)
            else:
                return 1.0 / 2.0

        def C3(psi):
            if psi > 1e-6:
                sp = np.sqrt(psi)
                return (sp - np.sin(sp)) / (psi * sp)
            elif psi < -1e-6:
                sp = np.sqrt(-psi)
                return (np.sinh(sp) - sp) / ((-psi) * sp)
            else:
                return 1.0 / 6.0

        # Newton-Raphson iteration with bisection fallback
        psi_n = 0.0
        psi_up = 4 * np.pi**2
        psi_low = -4 * np.pi**2

        for _ in range(100):
            c2 = C2(psi_n)
            c3 = C3(psi_n)

            y_n = r1 + r2 + A * (psi_n * c3 - 1) / np.sqrt(c2)

            if y_n < 0:
                # Readjust psi until y_n is non-negative (with iteration limit)
                for _ in range(2000):
                    psi_n += 0.1
                    c2 = C2(psi_n)
                    c3 = C3(psi_n)
                    y_n = r1 + r2 + A * (psi_n * c3 - 1) / np.sqrt(c2)
                    if y_n >= 0:
                        break
                else:
                    raise ValueError("Lambert solver: could not find valid y_n (geometry may be near-degenerate)")

            chi = np.sqrt(y_n / c2)

            tof_n = (chi**3 * c3 + A * np.sqrt(y_n)) / np.sqrt(self.mu)

            if abs(tof_n - tof) < 1e-8 * abs(tof):
                break

            if tof_n <= tof:
                psi_low = psi_n
            else:
                psi_up = psi_n

            # Newton step with bisection guard
            dtof_dpsi = (chi**3 * (C3(psi_n) - 3 * c3 * C2(psi_n) / (2 * c2)) / (2 * c2) +
                         (A / 8) * (3 * c3 * np.sqrt(y_n) / c2 + A / chi))
            dtof_dpsi /= np.sqrt(self.mu)

            if abs(dtof_dpsi) > 1e-14:
                psi_new = psi_n + (tof - tof_n) / dtof_dpsi
                if psi_low <= psi_new <= psi_up:
                    psi_n = psi_new
                else:
                    psi_n = (psi_up + psi_low) / 2
            else:
                psi_n = (psi_up + psi_low) / 2

        f = 1 - y_n / r1
        g_dot = 1 - y_n / r2
        g = A * np.sqrt(y_n / self.mu)

        if abs(g) < 1e-14:
            raise ValueError("Lambert solver: g is near zero")

        v1 = (r2_vec - f * r1_vec) / g
        v2 = (g_dot * r2_vec - r1_vec) / g

        return v1, v2


## 3. Asteroid Data (Table 3)

Orbital elements at epoch 2461000.5 (JD) for Earth, the two refueling asteroids (Ryugu, Bennu), and the five mining asteroids.

In [ ]:
# Base (Earth)
earth = OrbitalBody(
    name="Earth",
    a=1.0009, e=0.0173, i=0.0032,
    Omega=171.7283, omega=289.5838, M0=318.5855,
    epoch=0.0
)

# Refueling asteroids
ryugu = OrbitalBody(
    name="162173 Ryugu",
    a=1.1909, e=0.1911, i=5.8666,
    Omega=251.2915, omega=211.6168, M0=270.6594,
    epoch=0.0
)

bennu = OrbitalBody(
    name="101955 Bennu",
    a=1.1260, e=0.0204, i=6.0328,
    Omega=1.9690, omega=66.4073, M0=267.4691,
    epoch=0.0
)

# Mining asteroids
sg10 = OrbitalBody(
    name="2001 SG10",
    a=1.4487, e=0.4246, i=4.2568,
    Omega=184.8938, omega=101.6706, M0=340.8908,
    epoch=0.0
)

ml = OrbitalBody(
    name="1989 ML",
    a=1.2728, e=0.1369, i=4.3791,
    Omega=104.2721, omega=183.6253, M0=121.5130,
    epoch=0.0
)

fg3 = OrbitalBody(
    name="1996 FG3",
    a=1.0548, e=0.3501, i=1.9727,
    Omega=299.4710, omega=24.0570, M0=36.6506,
    epoch=0.0
)

cc21 = OrbitalBody(
    name="2001 CC21",
    a=1.0321, e=0.2192, i=4.8086,
    Omega=75.3575, omega=179.4026, M0=140.7404,
    epoch=0.0
)

anteros = OrbitalBody(
    name="1943 Anteros",
    a=1.4305, e=0.2559, i=8.7077,
    Omega=246.2935, omega=338.4366, M0=260.3741,
    epoch=0.0
)

# Store in lists
refueling_bodies = [ryugu, bennu]
mining_bodies = [sg10, ml, fg3, cc21, anteros]

print(f"Refueling asteroids ({len(refueling_bodies)}):")
for body in refueling_bodies:
    print(f"  - {body.name}")
print(f"\nMining asteroids ({len(mining_bodies)}):")
for body in mining_bodies:
    print(f"  - {body.name}")

## 4. Problem Parameters (Table 2)

In [ ]:
@dataclass
class Parameters:
    """Problem parameters from Table 2."""

    # Physical constants
    mu_sun: float = 1.0       # Gravitational parameter [AU^3/TU^2] (canonical)
    mu_earth: float = 3.986e5  # km^3/s^2
    g0: float = 9.81e-3       # km/s^2

    # Spacecraft
    m_dry: float = 300.0      # kg
    m_max: float = 20000.0    # kg
    q_max: float = 30.0       # kg
    I_sp: float = 457.0       # s

    # Problem size
    n_bv: int = 3             # Max spacecraft
    n_rv: int = 3             # Max refueling visits

    # Mission
    T_service: float = 2.0 / 58.132  # days to TU
    lambda_weight: float = 5e-5

    # Profit and mining (from case study)
    profit: float = 10.0      # Same for all
    mining_mass: float = 10.0  # kg, same for all

    # Parking orbit
    r0_park: float = 7000.0   # km

    # Unit conversions
    AU_to_km: float = 1.496e8
    TU_to_sec: float = 58.132 * 86400


params = Parameters()

print(f"Spacecraft:")
print(f"  Dry mass:    {params.m_dry} kg")
print(f"  Max mass:    {params.m_max} kg")
print(f"  Isp:         {params.I_sp} s")
print(f"\nMission:")
print(f"  Profit/asteroid: {params.profit}")
print(f"  Mining/asteroid: {params.mining_mass} kg")
print(f"  Lambda:          {params.lambda_weight}")

## 5. Index Sets & Node Mapping

Build index sets (Equations 1-9 from the paper) and map node indices to celestial bodies.

In [ ]:
def build_index_sets(params: Parameters, n_refuel: int, n_mine: int) -> Dict:
    """Build index sets (Equations 1-9)."""

    n_bv = params.n_bv
    n_rv = params.n_rv

    B0 = [0]
    Bv = list(range(1, n_bv + 1))
    Bs = list(range(0, n_bv + 1))
    Be = list(range(n_bv + 1, 2 * n_bv + 2))  # Fixed: start at n_bv+1 to avoid overlap with Bs

    R0 = list(range(2 * n_bv + 2, 2 * n_bv + n_refuel + 2))
    Rv = list(range(2 * n_bv + n_refuel + 2, 2 * n_bv + n_refuel * n_rv + 2))
    R = R0 + Rv

    M = list(range(2 * n_bv + n_refuel * n_rv + 2,
                   2 * n_bv + n_refuel * n_rv + n_mine + 2))

    V = R + M
    N = Bs + Be + V

    k_prime = {k: k + n_bv + 1 for k in Bs}  # Fixed: offset by n_bv+1 to match corrected Be

    return {
        'B0': B0, 'Bv': Bv, 'Bs': Bs, 'Be': Be,
        'R0': R0, 'Rv': Rv, 'R': R, 'M': M, 'V': V, 'N': N,
        'k_prime': k_prime
    }


def build_node_mapping(sets: Dict, refueling_bodies: List, mining_bodies: List) -> Tuple[Dict, Dict]:
    """Map node indices to celestial bodies."""

    node_to_body = {}
    node_to_name = {}

    # Bases (starting and ending -- both Earth)
    for node in sets['Bs'] + sets['Be']:
        node_to_body[node] = earth
        node_to_name[node] = "Earth"

    # Refueling (including virtual)
    for i, node in enumerate(sets['R']):
        original_idx = i % len(refueling_bodies)
        node_to_body[node] = refueling_bodies[original_idx]
        node_to_name[node] = refueling_bodies[original_idx].name

    # Mining
    for i, node in enumerate(sets['M']):
        node_to_body[node] = mining_bodies[i]
        node_to_name[node] = mining_bodies[i].name

    return node_to_body, node_to_name


In [ ]:
# Build for case study
sets = build_index_sets(params, n_refuel=2, n_mine=5)
node_to_body, node_to_name = build_node_mapping(sets, refueling_bodies, mining_bodies)

print(f"Bases (Bs): {sets['Bs']}")
print(f"Ending Bases (Be): {sets['Be']}")
print(f"Refueling (R): {sets['R']}")
print(f"Mining (M): {sets['M']}")

print(f"\nNode assignments:")
for node in sorted(node_to_name.keys())[:15]:
    print(f"  Node {node:2d}: {node_to_name[node]}")
if len(node_to_name) > 15:
    print("  ...")

## 6. Trajectory Optimizer (NLP)

Optimizes trajectories for single segments using Lambert's problem. Includes Earth departure/arrival delta-v corrections (Equation 48). The default NLP start follows the paper's lower-bound departure-time and Hohmann/previous-transfer-time logic; on first-seen asteroid-to-asteroid arcs, the optimizer also tries a small set of local wait seeds so the paper-route basin is not missed by the trust-region solver.


In [ ]:
class TrajectoryOptimizer:
    """Optimizes trajectory for a single segment using Lambert's problem."""

    def __init__(self, params: Parameters):
        self.params = params
        self.lambert = LambertSolver(mu=params.mu_sun)

    def compute_delta_v(self, body_i: OrbitalBody, body_j: OrbitalBody,
                        T_d: float, T_t: float) -> float:
        """
        Compute delta-v for a transfer.

        Returns delta-v in km/s.
        """
        r1 = body_i.position_at_time(T_d, self.params.mu_sun)
        r2 = body_j.position_at_time(T_d + T_t, self.params.mu_sun)
        v1_orbit = body_i.velocity_at_time(T_d, self.params.mu_sun)
        v2_orbit = body_j.velocity_at_time(T_d + T_t, self.params.mu_sun)

        try:
            v1_transfer, v2_transfer = self.lambert.solve(r1, r2, T_t, prograde=True)
        except Exception:
            return 100.0

        conversion = self.params.AU_to_km / self.params.TU_to_sec

        dv1_heli = np.linalg.norm(v1_transfer - v1_orbit) * conversion
        dv2_heli = np.linalg.norm(v2_orbit - v2_transfer) * conversion

        # Earth departure/arrival correction (Equation 48)
        if body_i.name == "Earth":
            v_inf = (v1_transfer - v1_orbit) * conversion
            dv1 = self._earth_departure_dv(v_inf)
        else:
            dv1 = dv1_heli

        if body_j.name == "Earth":
            v_inf = (v2_orbit - v2_transfer) * conversion
            dv2 = self._earth_arrival_dv(v_inf)
        else:
            dv2 = dv2_heli

        return dv1 + dv2

    def _earth_departure_dv(self, v_inf: np.ndarray) -> float:
        """Compute Earth departure delta-v (Equation 48)."""
        v_inf_mag = np.linalg.norm(v_inf)
        v_park = np.sqrt(self.params.mu_earth / self.params.r0_park)
        v_depart = np.sqrt(v_inf_mag**2 + 2 * self.params.mu_earth / self.params.r0_park)
        return abs(v_depart - v_park)

    def _earth_arrival_dv(self, v_inf: np.ndarray) -> float:
        """Compute Earth arrival delta-v (Equation 48)."""
        v_inf_mag = np.linalg.norm(v_inf)
        v_park = np.sqrt(self.params.mu_earth / self.params.r0_park)
        v_arrive = np.sqrt(v_inf_mag**2 + 2 * self.params.mu_earth / self.params.r0_park)
        return abs(v_arrive - v_park)

    def optimize_segment(self, body_i: OrbitalBody, body_j: OrbitalBody,
                         T_arrival_i: float, T_t_prev: float = None,
                         T_d_prev: float = None) -> Dict:
        """
        Optimize single trajectory segment using trust-region NLP (paper Sec. IV.B.2).

        Initial guess: T_d = T_d_min (lower bound, Eq. 44), T_t = T_t_prev (transfer
        time from previous iteration per paper Sec. IV.B.2, or Hohmann if unavailable).

        Returns:
        --------
        result : Dict with T_d, T_t, delta_v, T_a, mass_ratio
        """
        from scipy.optimize import Bounds as ScipyBounds
        # Service time at asteroid nodes; Earth is base depot so no hold (Eq. 44)
        service = self.params.T_service if body_i.name != "Earth" else 0.0
        T_d_min = T_arrival_i + service

        a_transfer = (body_i.a + body_j.a) / 2
        T_t_hoh = np.pi * np.sqrt(a_transfer**3 / self.params.mu_sun)

        # Paper Sec. IV.B.2: T_t warm-starts from the previous iteration when available.
        # On a first encounter, try both the classical Hohmann half-period and the
        # full-period seed; the latter matches the long-transfer basin reported in Table 5.
        if T_t_prev is not None:
            T_t_candidates = [T_t_prev]
        else:
            T_t_candidates = [T_t_hoh, 2.0 * T_t_hoh]

        # Paper Sec. IV.B.2 default: T_d starts at the lower bound (Eq. 44).
        # T_d_prev is only used by diagnostics that intentionally seed an exact trajectory.
        T_d_init = max(T_d_min, T_d_prev) if T_d_prev is not None else T_d_min
        initial_guesses = [(T_d_init, max(T_t_init, 1e-5)) for T_t_init in T_t_candidates]

        # The trust-region NLP is local. For first-seen asteroid-to-asteroid legs,
        # a short local-wait multistart recovers the FG3->Bennu basin reported in
        # Table 5 without changing Earth launch/return behavior.
        if T_t_prev is None and T_d_prev is None and body_i.name != "Earth" and body_j.name != "Earth":
            local_wait_offsets = [1.0, 2.0, 2.5, 3.0]
            local_transfer_seeds = [T_t_hoh, 2.0 * T_t_hoh, 5.0, 7.0, 9.0]
            for wait_offset in local_wait_offsets:
                for transfer_seed in local_transfer_seeds:
                    initial_guesses.append((T_d_min + wait_offset, max(transfer_seed, 1e-5)))

        bounds = ScipyBounds([T_d_min, 1e-5], [np.inf, np.inf])

        def objective(x):
            T_d, T_t = x
            dv = self.compute_delta_v(body_i, body_j, T_d, T_t)
            return dv if np.isfinite(dv) else 1e6

        best_res = None
        for T_d_start, T_t_init in initial_guesses:
            x0 = np.array([T_d_start, T_t_init], dtype=float)
            try:
                res = minimize(
                    objective,
                    x0=x0,
                    method='trust-constr',
                    bounds=bounds,
                    options={'maxiter': 500, 'verbose': 0, 'gtol': 1e-8, 'xtol': 1e-8, 'initial_tr_radius': 2}
                )
                # trust-constr often returns success=False even for valid solutions
                if res is not None and np.isfinite(res.fun) and res.fun < 100.0:
                    if best_res is None or res.fun < best_res.fun:
                        best_res = res
            except Exception:
                pass

        if best_res is None:
            fallback_d, fallback_t = initial_guesses[0]
            return {
                'T_d': fallback_d,
                'T_t': fallback_t,
                'delta_v': 100.0,
                'T_a': fallback_d + fallback_t,
                'mass_ratio': 1e-10
            }

        T_d_opt, T_t_opt = best_res.x
        dv_opt = best_res.fun
        mass_ratio = np.exp(-dv_opt / (self.params.g0 * self.params.I_sp))
        mass_ratio = min(max(mass_ratio, 1e-10), 0.999)

        return {
            'T_d': T_d_opt,
            'T_t': T_t_opt,
            'delta_v': dv_opt,
            'T_a': T_d_opt + T_t_opt,
            'mass_ratio': mass_ratio
        }

### Verification Against Table 5

Verify the orbital mechanics implementation against known trajectory data from the paper.

In [ ]:
print("--- Trajectory Verification (Paper Table 5) ---")
_traj_verify = TrajectoryOptimizer(Parameters())
_paper_legs = [
    ("Earth->FG3",   earth, fg3,   0.09, 6.26, 9.51),
    ("FG3->Bennu",   fg3,   bennu, 8.83, 7.06, 7.32),
    ("Bennu->Earth", bennu, earth, 17.59, 6.81, 8.17),
]
for name, bi, bj, Td, Tt, dv_expected in _paper_legs:
    dv_computed = _traj_verify.compute_delta_v(bi, bj, Td, Tt)
    err = abs(dv_computed - dv_expected) / dv_expected * 100
    status = "OK" if err < 15 else "MISMATCH"
    print(f"  {name:15s}: computed={dv_computed:.2f}, expected={dv_expected:.2f} km/s, err={err:.1f}% [{status}]")
print("--- End Verification ---")

### Diagnostic: dv Landscape & NLP Basin Analysis

Scan T_t at T_d=0 for key arcs to identify local minima and understand which basin
the NLP converges to from the Hohmann starting point.

In [ ]:

# ── Diagnostic: dv landscape scan for key arcs ───────────────────────────────
# Scans T_t at fixed T_d to reveal local minima in the dv surface.
# For Earth→FG3: paper uses T_d=0 as init starting point, finds T_t=6.26.
# Our NLP from same start finds T_t=4.41. This cell shows the full landscape.

_diag = TrajectoryOptimizer(Parameters())

def scan_dv_landscape(body_i, body_j, T_d_fixed, T_t_range=None, label=None):
    """Scan dv(T_t) at a fixed T_d and report local minima."""
    if T_t_range is None:
        T_t_range = np.linspace(0.5, 14.0, 300)
    dvs = []
    for T_t in T_t_range:
        try:
            dv = _diag.compute_delta_v(body_i, body_j, T_d_fixed, T_t)
            dvs.append(dv if np.isfinite(dv) and dv < 100 else np.nan)
        except Exception:
            dvs.append(np.nan)
    dvs = np.array(dvs)

    # Find local minima (simple: where dv[i] < dv[i-1] and dv[i] < dv[i+1])
    minima = []
    for i in range(1, len(dvs) - 1):
        if np.isfinite(dvs[i]) and np.isfinite(dvs[i-1]) and np.isfinite(dvs[i+1]):
            if dvs[i] < dvs[i-1] and dvs[i] < dvs[i+1]:
                minima.append((T_t_range[i], dvs[i]))

    # Hohmann T_t
    a_mid = (body_i.a + body_j.a) / 2
    T_t_hoh = np.pi * np.sqrt(a_mid**3 / 1.0)
    dv_hoh = _diag.compute_delta_v(body_i, body_j, T_d_fixed, T_t_hoh)

    if label:
        print(f"\n{'─'*70}")
        print(f"  Arc: {label}  |  T_d fixed = {T_d_fixed:.3f} TU")
        print(f"  Hohmann T_t = {T_t_hoh:.3f} TU  →  dv(Hohmann) = {dv_hoh:.4f} km/s")
        print(f"  Local minima found:")
        for tt, dv in minima:
            print(f"    T_t = {tt:.3f} TU  →  dv = {dv:.4f} km/s")

    return T_t_range, dvs, minima, T_t_hoh, dv_hoh


# ── 1. Earth → FG3 at T_d = 0 (paper init starting point) ──────────────────
T_t_vals, dvs_ef3, minima_ef3, T_t_hoh_ef3, dv_hoh_ef3 = scan_dv_landscape(
    earth, fg3, T_d_fixed=0.0, label="Earth → 1996 FG3 (T_d=0, paper init)"
)

# Also check gradient direction from Hohmann: does dv decrease toward T_t<Hohmann or T_t>Hohmann?
eps_grad = 0.05
dv_lo = _diag.compute_delta_v(earth, fg3, 0.0, T_t_hoh_ef3 - eps_grad)
dv_hi = _diag.compute_delta_v(earth, fg3, 0.0, T_t_hoh_ef3 + eps_grad)
grad = (dv_hi - dv_lo) / (2 * eps_grad)
print(f"\n  Numerical gradient d(dv)/d(T_t) at Hohmann: {grad:+.4f} km/s per TU")
print(f"    dv(T_t-ε) = {dv_lo:.4f},  dv(T_t+ε) = {dv_hi:.4f}")
print(f"    → NLP steepest descent moves T_t {'DOWN' if grad > 0 else 'UP'} from the Hohmann seed")

# ── 2. Earth → FG3 across a fine T_t grid: show where paper's basin (T_t≈6.26) sits ──
print(f"\n  Detailed scan near candidate minima:")
for T_t_check in [3.28, 4.0, 4.41, 5.0, 6.0, 6.26, 7.0, 8.0]:
    dv = _diag.compute_delta_v(earth, fg3, 0.0, T_t_check)
    marker = " ← PAPER" if abs(T_t_check - 6.26) < 0.1 else (" ← OUR NLP" if abs(T_t_check - 4.41) < 0.1 else (" ← HOHMANN" if abs(T_t_check - 3.28) < 0.1 else ""))
    print(f"    T_t={T_t_check:.2f}: dv={dv:.4f} km/s{marker}")

# ── 3. Earth → FG3 at T_d = T_d_min for iteration 1 (≈0 since Earth is base) ──
# T_d_min for Earth departure = 0 + 0 (no service time at Earth)
print(f"\n  Note: T_d_min for Earth departure = 0.0 (no service time at Earth base)")
print(f"  Init NLP and iteration-1 NLP both start at T_d=0 — landscape is IDENTICAL.")

# ── 4. FG3 → Bennu at T_d ≈ 4.47 (our NLP's arrival) vs T_d ≈ 6.35 (paper's arrival) ──
print(f"\n{'─'*70}")
print(f"  Cascade effect: FG3→Bennu landscape comparison")
_, _, minima_fb_ours, T_t_hoh_fb, _ = scan_dv_landscape(
    fg3, bennu, T_d_fixed=4.47, label="FG3 → Bennu (T_d=4.47 — our T_d_min)"
)
_, _, minima_fb_paper, _, _ = scan_dv_landscape(
    fg3, bennu, T_d_fixed=6.35, label="FG3 → Bennu (T_d=6.35 — paper's T_d_min)"
)

# ── 5. Summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("DIAGNOSTIC SUMMARY")
print(f"{'='*70}")
print(f"Earth→FG3 at T_d=0: paper finds T_t=6.26 (dv=9.51), we find T_t=4.41 (dv~10.2)")
print(f"  Both are local minima. Gradient at Hohmann T_t={T_t_hoh_ef3:.2f} determines which basin NLP enters.")
ef3_min_tt = min(minima_ef3, key=lambda x: x[1], default=(None,None))
print(f"  Global minimum in scan: T_t={ef3_min_tt[0]:.3f} TU, dv={ef3_min_tt[1]:.4f} km/s")
print(f"\nFix options:")
print(f"  Implemented test: try Hohmann half-period and full-period NLP starts, then keep the lower Δv local solution.")
print(f"  Remaining robust option: use a broader grid/multistart scan when the half/full starts still miss a basin.")


## 7. MILP Builder

Builds the Mixed-Integer Linear Program with fixed mass ratios. Implements the objective function (Eq. 10), network constraints (Eqs. 11-14), mass flow constraints (Eqs. 38-42), cumulative mining constraints (Eqs. 20-22), and physical limits (Eqs. 23-26).

In [ ]:
def build_milp(params: Parameters, sets: Dict, mass_ratios: Dict,
               node_to_name: Dict, node_to_body: Dict) -> Tuple[gp.Model, Dict]:
    """
    Build MILP model with fixed mass ratios.

    Returns model and variables dict.
    """

    model = gp.Model("VRTPP-PR")
    model.setParam('OutputFlag', 0)
    model.setParam('MIPGap', 0.0)  # Strict paper replication: prove optimal unless the 100 s limit is hit

    Bs, V, R, M = sets['Bs'], sets['V'], sets['R'], sets['M']
    k_prime = sets['k_prime']

    m_dry = params.m_dry
    m_max = params.m_max
    q_max = params.q_max
    lambda_w = params.lambda_weight
    m_m = params.mining_mass
    p = params.profit

    # Variables
    x, u, q, r, y = {}, {}, {}, {}, {}

    for k in Bs:
        for j in V:
            x[k, k, j] = model.addVar(vtype=GRB.BINARY)
    for k in Bs:
        for i in V:
            for j in V:
                # Filter same physical body: direct Bennu->Bennu (virtual) arcs
                # are physically meaningless and create near-free hops.
                if i != j and node_to_body[i].name != node_to_body[j].name:
                    x[k, i, j] = model.addVar(vtype=GRB.BINARY)
    for k in Bs:
        for i in V:
            x[k, i, k_prime[k]] = model.addVar(vtype=GRB.BINARY)

    for i in Bs + V:
        u[i] = model.addVar(lb=0, ub=m_max)
    for i in V:
        q[i] = model.addVar(lb=0, ub=q_max)
    for i in R:
        r[i] = model.addVar(lb=0)
    for k in Bs:
        for i in V:
            y[k, i] = model.addVar(lb=0, ub=q_max)

    model.update()

    # Objective (Eq. 10)
    profit_term = gp.quicksum(
        p * (gp.quicksum(x[k, i, j] for k in Bs for j in V if i != j) +
             gp.quicksum(x[k, i, k_prime[k]] for k in Bs))
        for i in M
    )
    fuel_term = gp.quicksum(u[k] - m_dry * gp.quicksum(x[k, k, j] for j in V) for k in Bs) + \
                gp.quicksum(r[i] for i in R)

    model.setObjective(profit_term - lambda_w * fuel_term, GRB.MAXIMIZE)

    # Network constraints (Eqs. 11-14)
    for k in Bs:
        model.addConstr(gp.quicksum(x[k, k, j] for j in V) <= 1)
    for j in R:
        model.addConstr(gp.quicksum(x[k, k, j] for k in Bs) +
                        gp.quicksum(x[k, i, j] for k in Bs for i in V if i != j and (k, i, j) in x) <= 1)
    for i in M:
        model.addConstr(gp.quicksum(x[k, i, j] for k in Bs for j in V if i != j) +
                        gp.quicksum(x[k, i, k_prime[k]] for k in Bs) <= 1)
    for j in V:
        for k in Bs:
            model.addConstr(x[k, k, j] - x[k, j, k_prime[k]] +
                            gp.quicksum(x[k, i, j] - x[k, j, i] for i in V if i != j and (k, i, j) in x) == 0)

    # Mass flow (Eqs. 38-42)
    # Upper bounds on arrival mass (mass conservation)
    for k in Bs:
        for j in V:
            if (k, j) in mass_ratios:
                m_kj = mass_ratios[(k, j)]
                model.addConstr(u[j] <= m_kj * u[k] + m_max * (1 - x[k, k, j]))

    for i in M:
        for j in V:
            if i != j and (i, j) in mass_ratios:
                m_ij = mass_ratios[(i, j)]
                if np.isfinite(m_ij) and 0 < m_ij <= 1:
                    model.addConstr(u[j] <= m_ij * (u[i] + m_m) + m_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))

    for i in R:
        for j in V:
            if i != j and (i, j) in mass_ratios:
                m_ij = mass_ratios[(i, j)]
                if np.isfinite(m_ij) and 0 < m_ij <= 1:
                    model.addConstr(u[j] <= m_ij * (u[i] + r[i]) + m_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))

    # Eqs. 41-42: Ending-base legs carry mined/refueled cargo (y[k,i] = q[i] on return)
    # Mining nodes -> ending base (Eq. 41)
    for i in M:
        for k in Bs:
            if (i, k_prime[k]) in mass_ratios:
                m_ik = mass_ratios[(i, k_prime[k])]
                model.addConstr(
                    m_dry + y[k, i] <= m_ik * (u[i] + m_m) + m_max * (1 - x[k, i, k_prime[k]])
                )

    # Refueling nodes -> ending base (Eq. 42)
    for i in R:
        for k in Bs:
            if (i, k_prime[k]) in mass_ratios:
                m_ik = mass_ratios[(i, k_prime[k])]
                model.addConstr(
                    m_dry + y[k, i] <= m_ik * (u[i] + r[i]) + m_max * (1 - x[k, i, k_prime[k]])
                )

    # Cumulative mining (Eqs. 20-22)
    for j in M:
        model.addConstr(q[j] >= m_m - q_max * (1 - gp.quicksum(x[k, k, j] for k in Bs)))
    for i in V:
        for j in M:
            if i != j:
                model.addConstr(q[j] >= q[i] + m_m - q_max * (1 - gp.quicksum(x[k, i, j] for k in Bs)))
    for i in V:
        for j in R:
            if i != j:
                model.addConstr(q[j] >= q[i] - q_max * (1 - gp.quicksum(x[k, i, j] for k in Bs if (k, i, j) in x)))

    # Physical limits (Eqs. 23-26)
    for i in M:
        model.addConstr(u[i] >= m_dry + q[i] - m_m)
        model.addConstr(u[i] + m_m <= m_max)
    for i in R:
        model.addConstr(u[i] >= m_dry + q[i])
        model.addConstr(u[i] + r[i] <= m_max)

    # Linearization: y[k,i] = q[i] * x[k,i,k'(k)] (cargo only on ending-base arc)
    # This tightens the paper definition: y_ki = q_i * x^k_{i,k'(k)}
    for k in Bs:
        for i in V:
            model.addConstr(y[k, i] <= q[i])
            model.addConstr(y[k, i] <= q_max * x[k, i, k_prime[k]])
            model.addConstr(y[k, i] >= q[i] - q_max * (1 - x[k, i, k_prime[k]]))

    return model, {'x': x, 'u': u, 'q': q, 'r': r, 'y': y}

## 8. Route Extraction & Mass Ratio Initialization

- `extract_routes`: Extracts spacecraft routes from the MILP binary variables.
- `initialize_mass_ratios`: Computes initial mass ratios per Section IV.A of the paper, using trust-region NLP starts at T_d=0 with both Hohmann half-period and full-period transfer-time seeds.

In [ ]:
def extract_routes(x_vars: Dict, sets: Dict) -> List[List[int]]:
    """
    Extract spacecraft routes from MILP binary solution.
    Reconstructs each vehicle's sequence: base -> asteroids -> ending_base.
    """
    Bs = sets['Bs']
    V = sets['V']
    k_prime = sets['k_prime']
    routes = []

    for k in Bs:
        # Find first arc out of starting base k
        start_node = None
        for j in V:
            var = x_vars.get((k, k, j))
            if var is not None:
                try:
                    if var.X > 0.5:
                        start_node = j
                        break
                except Exception:
                    pass

        if start_node is None:
            continue

        route = [k, start_node]
        current = start_node
        kp = k_prime[k]

        for _ in range(len(V) + 2):
            # Check if this node goes directly to ending base
            var_end = x_vars.get((k, current, kp))
            if var_end is not None:
                try:
                    if var_end.X > 0.5:
                        route.append(kp)
                        break
                except Exception:
                    pass

            # Find next asteroid
            next_node = None
            for j in V:
                if j == current:
                    continue
                var = x_vars.get((k, current, j))
                if var is not None:
                    try:
                        if var.X > 0.5:
                            next_node = j
                            break
                    except Exception:
                        pass

            if next_node is None:
                break
            route.append(next_node)
            current = next_node

        if len(route) >= 3:  # base -> ≥1 asteroid -> ending_base
            routes.append(route)

    return routes


In [ ]:
def initialize_mass_ratios(params: Parameters, sets: Dict, node_to_body: Dict) -> Tuple[Dict, Dict]:
    """
    Initialize mass ratios per paper Section IV.A.

    Paper states: "solve the trajectory optimization problem for each pair
    of bodies to find optimal departure and transfer times by using the zero
    departure time and the Hohmann transfer time as the initial guess."

    This implementation tries both the classical Hohmann half-period and the
    full-period seed, then keeps the lower-delta-v local NLP result.
    """
    print("Initializing mass ratios (T_d=0, Hohmann half/full-period NLP starts)...")

    traj_opt = TrajectoryOptimizer(params)
    mass_ratios = {}
    init_times = {}

    all_source = sets['Bs'] + sets['V']
    all_dest = sets['V'] + list(set(sets['Be']))

    body_pair_cache = {}
    body_pair_times = {}
    eps = 1e-5

    for i in all_source:
        for j in all_dest:
            if i == j:
                continue

            body_i = node_to_body[i]
            body_j = node_to_body[j]

            if body_i.name == body_j.name:
                continue

            pair_key = (body_i.name, body_j.name)
            if pair_key in body_pair_cache:
                mass_ratios[(i, j)] = body_pair_cache[pair_key]
                init_times[(i, j)] = body_pair_times[pair_key]
                continue

            a_transfer = (body_i.a + body_j.a) / 2
            T_t_hoh = np.pi * np.sqrt(a_transfer ** 3 / params.mu_sun)
            T_t_candidates = [T_t_hoh, 2.0 * T_t_hoh]
            best_td, best_tt = 0.0, T_t_candidates[0]
            best_dv = 1e6

            def _obj(x, bi=body_i, bj=body_j):
                T_d, T_t = x
                if T_t < eps:
                    return 1e6
                try:
                    return traj_opt.compute_delta_v(bi, bj, T_d, T_t)
                except Exception:
                    return 1e6

            from scipy.optimize import Bounds as _SB
            for T_t_init in T_t_candidates:
                try:
                    res = minimize(
                        _obj,
                        [0.0, T_t_init],
                        method='trust-constr',
                        bounds=_SB([0.0, eps], [np.inf, np.inf]),
                        options={'maxiter': 500, 'gtol': 1e-8, 'xtol': 1e-8,
                                 'verbose': 0, 'initial_tr_radius': 5}
                    )
                    if np.isfinite(res.fun) and 0 < res.fun < best_dv:
                        best_dv = float(res.fun)
                        best_td, best_tt = float(res.x[0]), float(res.x[1])
                except Exception:
                    pass

            if np.isfinite(best_dv) and 0 < best_dv < 50.0:
                mr = np.exp(-best_dv / (params.g0 * params.I_sp))
                mass_ratios[(i, j)] = float(np.clip(mr, 1e-4, 0.999))
            else:
                mass_ratios[(i, j)] = 0.05

            init_times[(i, j)] = (best_td, best_tt)
            body_pair_cache[pair_key] = mass_ratios[(i, j)]
            body_pair_times[pair_key] = (best_td, best_tt)

    valid_count = sum(1 for mr in mass_ratios.values() if np.isfinite(mr) and 0 < mr <= 1)
    print(f"  Initialized {len(mass_ratios)} transfers ({valid_count} valid)")

    mr_values = [v for v in mass_ratios.values() if v < 0.99]
    if mr_values:
        print(f"  Mass ratio range (excl same-body): [{min(mr_values):.4f}, {max(mr_values):.4f}]")

    return mass_ratios, init_times


In [ ]:
# ── Quick check: initialization mass ratios for ALL body pairs ───────────────
# Run this BEFORE the full solve to verify the initialization is sensible.
_init_mr, _init_times = initialize_mass_ratios(params, sets, node_to_body)

print("\n" + "=" * 75)
print("INITIALIZED MASS RATIOS BY UNIQUE BODY PAIR")
print("=" * 75)
print(f"{'From':<22} {'To':<22} {'Mass Ratio':>12} {'Est. dv (km/s)':>16}")
print("-" * 75)

# Deduplicate by body-pair name so we only show unique physical pairs
seen = set()
rows = []
for (i, j), mr in sorted(_init_mr.items()):
    bi = node_to_body[i].name if i in node_to_body else "?"
    bj = node_to_body[j].name if j in node_to_body else "?"
    key = (bi, bj)
    if key in seen or bi == bj:
        continue
    seen.add(key)
    dv_est = -np.log(max(mr, 1e-12)) * params.g0 * params.I_sp  # km/s (g0 in km/s^2)
    rows.append((bi, bj, mr, dv_est))

rows.sort(key=lambda x: x[3])  # sort by dv ascending
for bi, bj, mr, dv_est in rows:
    # Highlight paper's key legs
    flag = " <-- PAPER LEG" if (
        ("Earth" in bi and "FG3" in bj) or
        ("FG3" in bi and "Bennu" in bj) or
        ("Bennu" in bi and "Earth" in bj)
    ) else ""
    print(f"  {bi:<20} {bj:<20} {mr:>12.4f} {dv_est:>16.2f}{flag}")

print("\nPaper's expected dv values: Earth->FG3=9.51, FG3->Bennu=7.32, Bennu->Earth=8.17 km/s")


## 9. Iterative MILP-NLP Solver

The complete iterative algorithm that alternates between:
1. Solving the MILP with fixed mass ratios to determine optimal routes
2. Optimizing trajectories (NLP) along those routes to update mass ratios

Convergence is checked via Equation 47 (delta-v change) and route stability.

In [ ]:
def solve_vrtpp_pr(params: Parameters, sets: Dict, node_to_body: Dict,
                   node_to_name: Dict, max_iterations: int = 50,
                   convergence_tol: float = 1e-3) -> Dict:
    """
    Complete iterative MILP-NLP algorithm (paper Sec. IV.B).

    NLP warm-start per paper Sec. IV.B.2:
      T_d = T_d_min (lower bound, Eq. 44)
      T_t = transfer time from previous iteration; Hohmann half/full starts on first call

    Returns solution dict with routes, times, delta-v, etc.
    """

    print("=" * 80)
    print("STARTING VRTPP-PR OPTIMIZATION")
    print("=" * 80)

    traj_opt = TrajectoryOptimizer(params)

    # Step 1: Initialize mass ratios (paper Sec. IV.A)
    mass_ratios, init_times = initialize_mass_ratios(params, sets, node_to_body)
    delta_v_matrix = {}
    departure_times = {}
    transfer_times = {}
    arc_results = {}  # (i,j) -> last NLP result; used for T_t warm-start

    print("\nCritical mass ratios (Earth->FG3->Bennu->Earth):")
    for (i, j), mr in sorted(mass_ratios.items()):
        bi = node_to_body[i].name if i in node_to_body else "?"
        bj = node_to_body[j].name if j in node_to_body else "?"
        dv_est = -np.log(max(mr, 1e-10)) * params.g0 * params.I_sp
        if ("Earth" in bi and "FG3" in bj) or \
           ("FG3" in bi and "Bennu" in bj) or \
           ("Bennu" in bi and "Earth" in bj):
            print(f"  ({i:2d},{j:2d}) {bi:20s} -> {bj:20s}: mr={mr:.4f}, dv~{dv_est:.1f} km/s")

    warm_start = None
    prev_routes = None
    start_time = time.time()
    consecutive_no_routes = 0

    for iteration in range(max_iterations):
        print(f"\n{'=' * 80}")
        print(f"ITERATION {iteration + 1}")
        print(f"{'=' * 80}")

        # Step 2: Solve MILP with fixed mass ratios (paper Sec. IV.B.1)
        print("\n[MILP] Building model...")
        model, variables = build_milp(params, sets, mass_ratios, node_to_name, node_to_body)

        if warm_start:
            for key, val in warm_start.items():
                if key in variables['x']:
                    variables['x'][key].Start = val

        model.setParam('TimeLimit', 100.0)
        print("[MILP] Solving...")
        model.optimize()

        if model.Status == GRB.INFEASIBLE:
            print(f"[MILP] Infeasible (status {model.Status})")
            if iteration == 0:
                print("No feasible solution!")
                return None
            else:
                print("Using previous solution")
                break
        elif model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
            print(f"[MILP] Unexpected status {model.Status}")
            if iteration == 0:
                return None
            break

        if model.SolCount == 0:
            print(f"[MILP] No solution found (status {model.Status})")
            if iteration == 0:
                return None
            break

        if model.Status == GRB.TIME_LIMIT:
            print(f"[MILP] Time limit reached, using best solution (gap: {model.MIPGap*100:.1f}%)")

        print(f"[MILP] Objective: {model.ObjVal:.4f}")

        routes = extract_routes(variables['x'], sets)
        print(f"[MILP] Routes: {len(routes)} spacecraft")

        if len(routes) == 0:
            consecutive_no_routes += 1
            print(f"[MILP] No routes found ({consecutive_no_routes} consecutive)")
            if consecutive_no_routes >= 2:
                print("[MILP] Early termination: no routes for 2 consecutive iterations")
                break
            continue
        else:
            consecutive_no_routes = 0

        for i, route in enumerate(routes):
            route_names = [node_to_name[n] for n in route]
            print(f"  Spacecraft {i + 1}: {' -> '.join(route_names)}")

        # Step 3: Optimize trajectories (NLP, paper Sec. IV.B.2)
        # T_d warm-start = T_d_min (lower bound, Eq. 44)
        # T_t warm-start = T_t from previous iteration; Hohmann half/full starts if first call
        print("\n[NLP] Optimizing trajectories...")
        old_dv_matrix = delta_v_matrix.copy()

        current_arc_set = set()
        for spacecraft_route in routes:
            for k in range(len(spacecraft_route) - 1):
                current_arc_set.add((spacecraft_route[k], spacecraft_route[k + 1]))

        for spacecraft_route in routes:
            T_arrival = 0.0

            for k in range(len(spacecraft_route) - 1):
                node_i = spacecraft_route[k]
                node_j = spacecraft_route[k + 1]
                arc = (node_i, node_j)

                body_i = node_to_body[node_i]
                body_j = node_to_body[node_j]

                prev_result = arc_results.get(arc)
                # Paper Sec. IV.B.2: T_d starts at the lower bound (Eq. 44).
                # Only T_t is warm-started from the previous iteration; first calls try Hohmann half/full-period seeds.
                T_t_prev_val = prev_result['T_t'] if prev_result is not None else None

                print(f"  Optimizing {node_to_name[node_i]} -> {node_to_name[node_j]}...", end=" ")
                result = traj_opt.optimize_segment(body_i, body_j, T_arrival,
                                                   T_t_prev=T_t_prev_val)

                arc_results[arc] = result
                departure_times[arc] = result['T_d']
                transfer_times[arc] = result['T_t']
                delta_v_matrix[arc] = result['delta_v']
                mass_ratios[arc] = result['mass_ratio']
                T_arrival = result['T_a']

                # Propagate refined mass ratio to ALL virtual-node pairs of the same
                # physical body pair (paper uses a single m_ij per body pair, not per
                # virtual-node pair — different virtual nodes of Earth/asteroids must
                # share the same mass ratio or the MILP uses stale init values).
                for (si, sj) in list(mass_ratios.keys()):
                    if (si in node_to_body and sj in node_to_body and
                            node_to_body[si].name == body_i.name and
                            node_to_body[sj].name == body_j.name):
                        mass_ratios[(si, sj)] = result['mass_ratio']

                print(f"dv={result['delta_v']:.2f} km/s, T_d={result['T_d']:.2f} TU, T_t={result['T_t']:.2f} TU")

        # Step 4: Convergence check — Eq. 47: ||Δv_old - Δv_new||_F / max(Δv_old) ≤ ε_c
        # Use body-name pairs as keys to avoid node-index dependence on which spacecraft k
        # the MILP picks (different k values produce different arc (k,j) node pairs
        # for the same physical Earth→asteroid leg, breaking set intersection).
        def dv_by_body(dv_dict):
            result_dict = {}
            for (ni, nj), dv in dv_dict.items():
                bi = node_to_body[ni].name if ni in node_to_body else f'base_{ni}'
                bj = node_to_body[nj].name if nj in node_to_body else f'base_{nj}'
                result_dict[(bi, bj)] = dv
            return result_dict

        def route_sig(rts):
            return frozenset(tuple(node_to_body[n].name for n in r) for r in rts)

        route_names = [' -> '.join(node_to_name[n] for n in r) for r in routes]
        if iteration > 0 and old_dv_matrix:
            route_stable = (prev_routes is not None and
                            route_sig(routes) == route_sig(prev_routes))
            if not route_stable:
                print(f"\n[CONVERGENCE] Route changed → resetting.")
                print(f"  New routes: {route_names}")
            else:
                old_body = dv_by_body(old_dv_matrix)
                new_body = dv_by_body(delta_v_matrix)
                common = set(old_body.keys()) & set(new_body.keys())
                diff_sq = sum((new_body[k] - old_body[k]) ** 2 for k in common)
                max_dv_old = max(
                    (v for v in old_body.values() if np.isfinite(v) and v > 0),
                    default=1.0
                )
                change = np.sqrt(diff_sq) / max_dv_old
                print(f"\n[CONVERGENCE] Frobenius norm (Eq.47): {change:.6f} (tol: {convergence_tol})")
                print(f"  Routes: {route_names}")
                if change < convergence_tol:
                    print(f"\n{'=' * 80}")
                    print(f"CONVERGED after {iteration + 1} iterations!")
                    print(f"{'=' * 80}")
                    break

        prev_routes = [r[:] for r in routes]

        warm_start = {}
        for k, v in variables['x'].items():
            try:
                if v.X > 0.5:
                    warm_start[k] = v.X
            except Exception:
                continue

    elapsed = time.time() - start_time

    u_values = {}
    r_values = {}
    q_values = {}
    if model.SolCount > 0:
        for key, var in variables['u'].items():
            try:
                u_values[key] = var.X
            except Exception:
                u_values[key] = 0.0
        for key, var in variables['r'].items():
            try:
                r_values[key] = var.X
            except Exception:
                r_values[key] = 0.0
        for key, var in variables['q'].items():
            try:
                q_values[key] = var.X
            except Exception:
                q_values[key] = 0.0

    solution = {
        'status': 'converged' if iteration < max_iterations - 1 else 'max_iterations',
        'iterations': iteration + 1,
        'elapsed_time': elapsed,
        'objective': model.ObjVal if model.SolCount > 0 else 0.0,
        'routes': routes,
        'departure_times': departure_times,
        'transfer_times': transfer_times,
        'delta_v_matrix': delta_v_matrix,
        'mass_ratios': mass_ratios,
        'u_values': u_values,
        'r_values': r_values,
        'q_values': q_values
    }

    return solution

## 10. Run Optimization

In [ ]:
solution = solve_vrtpp_pr(
    params=params,
    sets=sets,
    node_to_body=node_to_body,
    node_to_name=node_to_name,
    max_iterations=50,
    convergence_tol=1e-3
)

In [ ]:
# ── Diagnostic: All mass ratios and transfer times ──────────────────────────
if not solution:
    print("No solution available.")
else:
    # 1. ALL MASS RATIOS (from initialization + NLP updates)
    print("=" * 85)
    print("ALL MASS RATIOS  (mr = exp(-dv / (g0*Isp)), dv estimated from mr)")
    print("=" * 85)
    print(f"{'From':<22} {'To':<22} {'Mass Ratio':>12} {'Est. dv (km/s)':>16} {'Source':>10}")
    print("-" * 85)

    mr_rows = []
    for (i, j), mr in sorted(solution['mass_ratios'].items()):
        name_i = node_to_name.get(i, f"node{i}")
        name_j = node_to_name.get(j, f"node{j}")
        dv_est = -np.log(max(mr, 1e-12)) * params.g0 * params.I_sp  # km/s (g0 in km/s^2)
        # Flag NLP-refined legs (they appear in delta_v_matrix)
        source = "NLP" if (i, j) in solution['delta_v_matrix'] else "init"
        mr_rows.append((name_i, name_j, mr, dv_est, source))

    # Sort by source (NLP first) then by dv
    mr_rows.sort(key=lambda x: (x[4] != "NLP", x[3]))
    for name_i, name_j, mr, dv_est, source in mr_rows:
        print(f"  {name_i:<20} {name_j:<20} {mr:>12.4f} {dv_est:>16.2f} {source:>10}")

    # 2. NLP-REFINED LEGS ONLY (actual optimized values)
    print("\n" + "=" * 85)
    print("NLP-REFINED TRAJECTORY SEGMENTS (actual optimized values)")
    print("=" * 85)
    print(f"{'Segment':<40} {'T_dep (TU)':>12} {'T_trans (TU)':>14} {'dv (km/s)':>12} {'Mass Ratio':>12}")
    print("-" * 85)
    for (i, j), dv in sorted(solution['delta_v_matrix'].items(), key=lambda x: solution['departure_times'].get(x[0], 0)):
        name_i = node_to_name.get(i, f"node{i}")
        name_j = node_to_name.get(j, f"node{j}")
        T_d = solution['departure_times'].get((i, j), float('nan'))
        T_t = solution['transfer_times'].get((i, j), float('nan'))
        mr = solution['mass_ratios'].get((i, j), float('nan'))
        seg = f"{name_i} -> {name_j}"
        print(f"  {seg:<38} {T_d:>12.4f} {T_t:>14.4f} {dv:>12.4f} {mr:>12.4f}")

    # 3. PAPER TABLE 5 COMPARISON
    print("\n" + "=" * 85)
    print("PAPER TABLE 5 REFERENCE VALUES")
    print("=" * 85)
    _ref = [
        ("Earth -> 1996 FG3",  0.09, 6.26, 9.51),
        ("1996 FG3 -> 101955 Bennu", 8.83, 7.06, 7.32),
        ("101955 Bennu -> Earth", 17.59, 6.81, 8.17),
    ]
    print(f"{'Segment':<40} {'T_dep (TU)':>12} {'T_trans (TU)':>14} {'dv (km/s)':>12}")
    print("-" * 85)
    for seg, td, tt, dv in _ref:
        print(f"  {seg:<38} {td:>12.2f} {tt:>14.2f} {dv:>12.2f}")


## 11. Results Display

In [ ]:
if solution and solution.get('routes'):
    print("=" * 80)
    print("FINAL SOLUTION")
    print("=" * 80)

    print(f"\nStatus: {solution['status']}")
    print(f"Iterations: {solution['iterations']}")
    print(f"Computation time: {solution['elapsed_time']:.2f} seconds")
    print(f"Objective value: {solution['objective']:.4f}")

    # Routes with detailed trajectory info
    print("\n" + "-" * 80)
    print("ROUTES AND TRAJECTORIES")
    print("-" * 80)

    for i, route in enumerate(solution['routes']):
        print(f"\nSpacecraft {i + 1}:")
        print(f"  Route: {' -> '.join([node_to_name[n] for n in route])}")
        print(f"\n  Trajectory segments:")

        for j in range(len(route) - 1):
            node_i, node_j = route[j], route[j + 1]

            if (node_i, node_j) in solution['delta_v_matrix']:
                T_d = solution['departure_times'][(node_i, node_j)]
                T_t = solution['transfer_times'][(node_i, node_j)]
                dv = solution['delta_v_matrix'][(node_i, node_j)]

                print(f"    {node_to_name[node_i]} -> {node_to_name[node_j]}:")
                print(f"      Departure time: {T_d:.2f} TU ({T_d * 58.132:.0f} days)")
                print(f"      Transfer time:  {T_t:.2f} TU ({T_t * 58.132:.0f} days)")
                print(f"      Delta-v:        {dv:.2f} km/s")

    # Mass and fuel details
    print("\n" + "-" * 80)
    print("MASS AND FUEL")
    print("-" * 80)

    u_vals = solution['u_values']
    r_vals = solution['r_values']

    # Compute initial propellant from starting bases
    initial_fuel = 0.0
    print("\n  Starting base masses:")
    for k in sets['Bs']:
        u_val = u_vals.get(k, 0.0)
        deployed = any(k == route[0] for route in solution['routes'])
        print(f"    u[{k}] = {u_val:.2f} kg (deployed: {deployed})")
        if u_val > params.m_dry and deployed:
            initial_fuel += u_val - params.m_dry

    total_refuel = sum(r_vals.get(i, 0.0) for i in sets['R'])

    print(f"\nInitial propellant:  {initial_fuel:8.2f} kg")
    print(f"Total refueling:     {total_refuel:8.2f} kg")
    print(f"Total fuel consumed: {initial_fuel + total_refuel:8.2f} kg")

    # Refueling details
    print(f"\nRefueling amounts:")
    for i in sets['R']:
        r_val = r_vals.get(i, 0.0)
        if r_val > 0.1:
            print(f"  {node_to_name[i]:20s}: {r_val:8.2f} kg")

    # Mass along routes
    print(f"\n  Mass at each node:")
    for route in solution['routes']:
        names = [node_to_name[n] for n in route]
        masses = [u_vals.get(n, 0.0) for n in route]
        print(f"    Route: {' -> '.join(names)}")
        for n, name, mass in zip(route, names, masses):
            extra = ""
            if n in sets['R']:
                r_val = r_vals.get(n, 0.0)
                if r_val > 0.1:
                    extra = f" (refuels {r_val:.1f} kg)"
            print(f"      {name:20s}: u={mass:.2f} kg{extra}")

    # Mining summary
    total_mined = sum(
        params.mining_mass
        for route in solution['routes']
        for node in route
        if node in sets['M']
    )
    print(f"\nTotal mined: {total_mined:.2f} kg")

    # Mission duration
    max_time = 0
    for route in solution['routes']:
        for j in range(len(route) - 1):
            if (route[j], route[j + 1]) in solution['transfer_times']:
                route_time = (solution['departure_times'][(route[j], route[j + 1])] +
                              solution['transfer_times'][(route[j], route[j + 1])])
                max_time = max(max_time, route_time)

    print(f"\nMission duration: {max_time:.2f} TU ({max_time * 58.132:.0f} days, {max_time * 58.132 / 365.25:.2f} years)")
else:
    print("\nNo routes found in solution.")

## 12. Visualization

Plot the mission routes showing each spacecraft's trajectory through the asteroid network, with delta-v labels on each segment.

In [ ]:
if solution and solution.get('routes') and len(solution['routes']) > 0:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for sc_idx, route in enumerate(solution['routes']):
        y_pos = sc_idx
        color = colors[sc_idx % len(colors)]

        for step, node in enumerate(route):
            if node in sets['Bs'] + sets['Be']:
                marker, size, node_color = 's', 250, 'blue'
            elif node in sets['R']:
                marker, size, node_color = '^', 180, 'orange'
            else:
                marker, size, node_color = 'o', 180, 'green'

            ax.scatter(step, y_pos, s=size, marker=marker,
                       color=node_color, edgecolors='black', linewidths=2, zorder=3)
            ax.text(step, y_pos - 0.18, node_to_name[node],
                    ha='center', va='top', fontsize=9, fontweight='bold')

            if step > 0:
                prev_node = route[step - 1]
                ax.plot([step - 1, step], [y_pos, y_pos], color=color, linewidth=3, zorder=1)
                if (prev_node, node) in solution['delta_v_matrix']:
                    dv = solution['delta_v_matrix'][(prev_node, node)]
                    ax.text((step - 1 + step) / 2, y_pos + 0.08,
                            f"{dv:.1f} km/s", ha='center', fontsize=8,
                            color=color, weight='bold')

    ax.set_xlabel('Step', fontsize=13, fontweight='bold')
    ax.set_ylabel('Spacecraft', fontsize=13, fontweight='bold')
    ax.set_title('Asteroid Mining Mission Routes\n(VRTPP-PR Paper Model — Case Study)',
                 fontsize=15, fontweight='bold', pad=20)
    ax.set_yticks(range(len(solution['routes'])))
    ax.set_yticklabels([f'Spacecraft {i + 1}' for i in range(len(solution['routes']))])
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(-0.5, len(solution['routes']) - 0.5)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue',   edgecolor='black', label='Base (Earth)'),
        Patch(facecolor='orange', edgecolor='black', label='Refueling Asteroid'),
        Patch(facecolor='green',  edgecolor='black', label='Mining Asteroid'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.savefig('mission_routes_papermodel.png', dpi=150, bbox_inches='tight')
    print("Plot saved to mission_routes_papermodel.png")
    plt.show()


## 13. Porkchop Plots (ΔV Contours per Route Leg)

Reproduces **Figure 2** from the paper. For each transfer segment in the optimal route, sweeps over a grid of departure times and transfer times to produce a ΔV porkchop plot.  
- **Color**: total Δv (km/s)  
- **Red ×**: selected (optimized) trajectory  
- **Dashed white line**: earliest allowable departure time (previous arrival + service time)

In [ ]:
if solution and solution.get('routes') and len(solution['routes']) > 0:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker

    _traj_opt = TrajectoryOptimizer(params)

    # Collect legs from all spacecraft routes
    legs = []
    for route in solution['routes']:
        T_arrival = 0.0
        for k in range(len(route) - 1):
            ni, nj = route[k], route[k + 1]
            bi, bj = node_to_body[ni], node_to_body[nj]
            T_d_sel = solution['departure_times'].get((ni, nj), T_arrival)
            T_t_sel = solution['transfer_times'].get((ni, nj), 7.0)
            T_d_min = T_arrival + params.T_service
            legs.append(dict(
                body_i=bi, body_j=bj,
                name_i=node_to_name[ni], name_j=node_to_name[nj],
                T_d_sel=T_d_sel, T_t_sel=T_t_sel,
                T_d_min=T_d_min,
            ))
            T_arrival = T_d_sel + T_t_sel

    n_legs = len(legs)
    fig, axes = plt.subplots(1, n_legs, figsize=(7 * n_legs, 6))
    if n_legs == 1:
        axes = [axes]

    N_td, N_tt = 60, 60
    panel_labels = 'abcdefghijklmnop'

    for ax, leg, lbl in zip(axes, legs, panel_labels):
        bi, bj = leg['body_i'], leg['body_j']
        T_d_min = leg['T_d_min']
        T_d_sel = leg['T_d_sel']
        T_t_sel = leg['T_t_sel']

        x_lo = max(0.0, T_d_min - 1.5)
        x_hi = x_lo + 16.0
        if T_d_sel > x_hi - 0.5:
            x_hi = T_d_sel + 2.0

        T_d_arr = np.linspace(x_lo, x_hi, N_td)
        T_t_arr = np.linspace(1.0, 16.0, N_tt)
        TT_d, TT_t = np.meshgrid(T_d_arr, T_t_arr)

        DV = np.full_like(TT_d, np.nan)
        for ii in range(N_tt):
            for jj in range(N_td):
                try:
                    dv = _traj_opt.compute_delta_v(bi, bj, TT_d[ii, jj], TT_t[ii, jj])
                    DV[ii, jj] = dv if dv < 99.0 else np.nan
                except Exception:
                    pass

        dv_max = np.nanpercentile(DV, 97)
        dv_min = np.nanmin(DV)

        im = ax.pcolormesh(TT_d, TT_t, DV, cmap='viridis', shading='auto',
                           vmin=dv_min, vmax=dv_max)
        levels = np.linspace(dv_min, dv_max, 12)
        ax.contour(TT_d, TT_t, DV, levels=levels, colors='cyan',
                   linewidths=0.6, alpha=0.55)

        cb = plt.colorbar(im, ax=ax, pad=0.02)
        cb.set_label('Velocity Change [km/s]', fontsize=10)
        cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))

        ax.axvline(x=T_d_min, color='white', linestyle='--', linewidth=1.5,
                   label='Earliest Departure Time')
        ax.scatter(T_d_sel, T_t_sel, marker='x', color='red', s=180,
                   linewidths=2.5, zorder=5, label='Selected Trajectory')

        ax.set_xlabel('Departure Time [TU]', fontsize=11)
        ax.set_ylabel('Transfer Time [TU]', fontsize=11)
        ax.set_title(f'{leg["name_i"]} \u2192 {leg["name_j"]}',
                     fontsize=11, fontweight='bold')
        ax.legend(loc='upper right', fontsize=8, framealpha=0.75,
                  handletextpad=0.4, borderpad=0.4)
        ax.text(0.03, 0.97, f'{lbl})', transform=ax.transAxes,
                fontsize=13, fontweight='bold', color='white',
                va='top', ha='left')

    fig.suptitle('\u0394V Porkchop Plots \u2014 Optimal Route Segments (VRTPP-PR Paper Model)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('porkchop_plots_papermodel.png', dpi=150, bbox_inches='tight')
    print('Saved: porkchop_plots_papermodel.png')
    plt.show()
else:
    print('No solution available — run the optimisation cells first.')


## 14. Experiment 3 - Paper Route Arc Verification

Directly verifies whether our trajectory optimizer can reproduce the paper's Table 5 values for Earth -> FG3 -> Bennu -> Earth. Three tests:
- **Part 1 (Grid search):** Coarse grid scan over (T_d, T_t) to find the minimum of `compute_delta_v` independent of NLP warm-start.
- **Part 2 (NLP, paper-context multistart):** `optimize_segment` from T_d_min with Hohmann half/full-period seeds plus the local asteroid-to-asteroid wait guard, using the paper's T_arrival values to set the correct T_d_min context.
- **Part 3 (NLP, exact paper seed):** `optimize_segment` seeded with the paper's exact (T_d, T_t), testing whether the NLP can converge to the paper's values given the right starting basin.


In [ ]:
# Experiment 3: Paper Route Verification
print("=" * 70)
print("EXPERIMENT 3: Paper Route Verification")
print("Earth -> 1996 FG3 -> 101955 Bennu -> Earth")
print("=" * 70)

# Paper reference values (Table 5)
paper_arcs = [
    ("Earth",        "1996 FG3",      0.09,  6.26, 9.51),
    ("1996 FG3",     "101955 Bennu",  8.83,  7.06, 7.32),
    ("101955 Bennu", "Earth",        17.59,  6.81, 8.17),
]

# T_arrival at each body using paper's exact timing
T_arrivals_paper = {
    "Earth":         0.0,
    "1996 FG3":      0.09 + 6.26,   # 6.35 TU
    "101955 Bennu":  8.83 + 7.06,   # 15.89 TU
}

traj_exp = TrajectoryOptimizer(params)
all_bodies = [earth] + refueling_bodies + mining_bodies
body_map = {b.name: b for b in all_bodies}

# Part 1: Grid search (independent of NLP) ----------------------------------
print()
print("--- Part 1: Grid search over (T_d, T_t) ---")
print(f"{'Arc':<34} {'Paper':>8} {'Grid':>8} {'T_d':>7} {'T_t':>7}")
print("-" * 68)

for src, dst, td_p, tt_p, dv_p in paper_arcs:
    bi, bj = body_map[src], body_map[dst]
    best_dv, best_td, best_tt = 1e9, td_p, tt_p
    for td in np.arange(0.0, 22.0, 0.5):
        for tt in np.arange(0.5, 14.0, 0.5):
            try:
                dv = traj_exp.compute_delta_v(bi, bj, td, tt)
                if np.isfinite(dv) and dv < best_dv:
                    best_dv, best_td, best_tt = dv, td, tt
            except Exception:
                pass
    label = f"{src} -> {dst}"
    match = "OK" if abs(best_dv - dv_p) < 0.2 else "DIFF"
    print(f"{label:<34} {dv_p:>8.3f} {best_dv:>8.3f} {best_td:>7.2f} {best_tt:>7.2f}  [{match}]")

# Part 2: NLP with paper-context multistart -------------------------------
print()
print("--- Part 2: NLP with paper-context multistart from T_d_min ---")
print(f"{'Arc':<34} {'Paper':>8} {'NLP':>8} {'T_d':>7} {'T_t':>7}")
print("-" * 68)

for src, dst, td_p, tt_p, dv_p in paper_arcs:
    bi, bj = body_map[src], body_map[dst]
    T_arr = T_arrivals_paper[src]
    res = traj_exp.optimize_segment(bi, bj, T_arr)
    label = f"{src} -> {dst}"
    match = "OK" if abs(res['delta_v'] - dv_p) < 0.2 else "DIFF"
    print(f"{label:<34} {dv_p:>8.3f} {res['delta_v']:>8.3f} {res['T_d']:>7.3f} {res['T_t']:>7.3f}  [{match}]")

# Part 3: NLP seeded with paper's exact (T_d, T_t) --------------------------
print()
print("--- Part 3: NLP seeded from paper's exact (T_d, T_t) ---")
print(f"{'Arc':<34} {'Paper':>8} {'NLP':>8} {'T_d':>7} {'T_t':>7}")
print("-" * 68)

for src, dst, td_p, tt_p, dv_p in paper_arcs:
    bi, bj = body_map[src], body_map[dst]
    T_arr = T_arrivals_paper[src]
    res = traj_exp.optimize_segment(bi, bj, T_arr, T_d_prev=td_p, T_t_prev=tt_p)
    label = f"{src} -> {dst}"
    match = "OK" if abs(res['delta_v'] - dv_p) < 0.2 else "DIFF"
    print(f"{label:<34} {dv_p:>8.3f} {res['delta_v']:>8.3f} {res['T_d']:>7.3f} {res['T_t']:>7.3f}  [{match}]")

print()
print("OK = within 0.2 km/s of paper; DIFF = larger gap")
